In [13]:
"""
Daily price fetcher:
- yfinance: THYAO, LKMNH, ASML, HLAL, SPUS + USDTRY + Gold + Silver
- tefas (selenium-based): KPC, HKH, RBH (TEFAS funds)

Install (in your conda env):
  pip install yfinance tefas selenium webdriver-manager

Notes:
- yfinance: prefers fast_info["lastPrice"] (more reliable than .info)
- tefas: launches Chrome via selenium; make sure Chrome is installed
"""

from __future__ import annotations

import json
from datetime import datetime, timezone
from typing import Any, Dict, Optional

import yfinance as yf

# tefas (PyPI package name: tefas)
import tefas


TROY_OUNCE_GRAMS = 31.1035


def _now_iso_utc() -> str:
    return datetime.now(timezone.utc).isoformat()


def yf_last_price(symbol: str) -> Dict[str, Any]:
    """
    Fetch latest price from yfinance using fast_info first (preferred),
    then fallback to .info["regularMarketPrice"] if needed.
    """
    t = yf.Ticker(symbol)

    price: Optional[float] = None
    currency: Optional[str] = None

    # Preferred: fast_info
    try:
        fi = t.fast_info
        if fi:
            # keys can vary; lastPrice is typical
            p = fi.get("lastPrice", None)
            if p is None:
                # fallback keys sometimes exist
                p = fi.get("last_price", None) or fi.get("last", None)
            if p is not None:
                price = float(p)

            c = fi.get("currency", None)
            if c:
                currency = str(c)
    except Exception:
        pass

    # Fallback: .info (slower / less reliable)
    if price is None:
        info = t.info
        p = info.get("regularMarketPrice", None)
        if p is None:
            p = info.get("previousClose", None)
        if p is None:
            raise ValueError(f"Could not get price for {symbol} from yfinance.")
        price = float(p)

        if currency is None:
            currency = info.get("currency", None)

    return {
        "symbol": symbol,
        "price": price,
        "currency": currency,
        "asof_utc": _now_iso_utc(),
        "source": "yfinance",
    }


def gram_try_from_usd_oz(usd_per_oz: float, usdtry: float) -> float:
    return (usd_per_oz * usdtry) / TROY_OUNCE_GRAMS


def tefas_latest_price(fon_kodu: str) -> Dict[str, Any]:
    """
    Fetch latest available TEFAS fund unit price via selenium-based tefas.get_data().
    Returns last row value for the fund column.

    tefas.get_data returns a pandas DataFrame; we only take the latest scalar.
    """
    df = tefas.get_data(fon_kodu, verbose=False)

    # Column name is typically the fund code itself
    if fon_kodu not in df.columns:
        # Try any column that matches ignoring case
        cols = {c.upper(): c for c in df.columns}
        if fon_kodu.upper() in cols:
            col = cols[fon_kodu.upper()]
        else:
            raise ValueError(f"TEFAS dataframe has no column for {fon_kodu}. Columns: {list(df.columns)}")
    else:
        col = fon_kodu

    latest_val = df[col].iloc[-1]
    # Index is often datetime-like; keep string-safe
    latest_date = str(df.index[-1])

    return {
        "symbol": fon_kodu,
        "price": float(latest_val),
        "currency": "TRY",
        "date": latest_date,
        "asof_utc": _now_iso_utc(),
        "source": "tefas(selenium)",
    }


def fetch_all_prices() -> Dict[str, Any]:
    out: Dict[str, Any] = {}

    # --- Your stocks / ETFs via yfinance ---
    yf_symbols = {
        "THYAO": "THYAO.IS",
        "LKMNH": "LKMNH.IS",
        "ASML": "ASML",
        "HLAL": "HLAL",
        "SPUS": "SPUS",
    }

    for name, sym in yf_symbols.items():
        out[name] = yf_last_price(sym)

    # --- FX ---
    usdtry = yf_last_price("USDTRY=X")
    out["USDTRY"] = usdtry

    usdtry_val = usdtry["price"]

    # --- Metals (USD/oz) via futures on Yahoo ---
    # Gold futures: GC=F, Silver futures: SI=F
    gold_oz = yf_last_price("GC=F")
    silver_oz = yf_last_price("SI=F")
    out["GOLD_OZ_USD"] = gold_oz
    out["SILVER_OZ_USD"] = silver_oz

    # --- Derived gram metals in TRY ---
    out["GOLD_GRAM_TRY"] = {
        "symbol": "XAUTRY(gram)",
        "price": gram_try_from_usd_oz(gold_oz["price"], usdtry_val),
        "currency": "TRY",
        "asof_utc": _now_iso_utc(),
        "source": "derived(GC=F * USDTRY=X / 31.1035)",
    }
    out["SILVER_GRAM_TRY"] = {
        "symbol": "XAGTRY(gram)",
        "price": gram_try_from_usd_oz(silver_oz["price"], usdtry_val),
        "currency": "TRY",
        "asof_utc": _now_iso_utc(),
        "source": "derived(SI=F * USDTRY=X / 31.1035)",
    }

    # --- TEFAS funds via selenium-based tefas ---
    for fund in ["KPC", "HKH", "RBH"]:
        out[fund] = tefas_latest_price(fund)

    return out


if __name__ == "__main__":
    prices = fetch_all_prices()

    # Pretty print
    for k, v in prices.items():
        print(f"{k:16s} => {v.get('price')} {v.get('currency') or ''} | {v.get('source')}")

    # Optional: save JSON (uncomment if you want)
    # with open("daily_prices.json", "w", encoding="utf-8") as f:
    #     json.dump(prices, f, ensure_ascii=False, indent=2)


THYAO            => 336.25 TRY | yfinance
LKMNH            => 17.34000015258789 TRY | yfinance
ASML             => 1435.6300048828125 USD | yfinance
HLAL             => 63.459999084472656 USD | yfinance
SPUS             => 51.900001525878906 USD | yfinance
USDTRY           => 43.6432991027832 TRY | yfinance
GOLD_OZ_USD      => 5072.7001953125 USD | yfinance
SILVER_OZ_USD    => 81.86499786376953 USD | yfinance
GOLD_GRAM_TRY    => 7117.828279221638 TRY | derived(GC=F * USDTRY=X / 31.1035)
SILVER_GRAM_TRY  => 114.86998530124268 TRY | derived(SI=F * USDTRY=X / 31.1035)
KPC              => 17.813704 TRY | tefas(selenium)
HKH              => 8.90061 TRY | tefas(selenium)
RBH              => 29.044579 TRY | tefas(selenium)
